# v05 · Two defects in HyperGallery, each demonstrated by brute force

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


## 1. `spectral_address_range` (the JWST search function)
It returns one interval `[addr_mid − δ, addr_mid + δ]` meant to bound every image whose bands lie in given ranges. Checked exhaustively on a spec small enough to enumerate: 2×2 pixels, 2 bands, 2-bit depth (4⁸ = 65,536 images).

In [2]:
import itertools
hg = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hypergallery.py", "hg")
s = hg.ImageSpec(2, 2, hg.PixelMode.SPECTRAL, n_bands=2, bit_depth=2)
G = hg.HyperGallery(s)
cons = {0: (1, 2)}                       # band 0 in [1,2] at every pixel
lo, hi = G.spectral_address_range(cons)
sat, inr = [], []
for v in itertools.product(range(4), repeat=s.n_values):
    a = G._pixels_to_int(list(v))
    ok = all(cons[0][0] <= v[px*2] <= cons[0][1] for px in range(4))
    if ok: sat.append(a)
    if lo <= a <= hi: inr.append(ok)
runs = 1 + sum(1 for x, y in zip(sorted(sat), sorted(sat)[1:]) if y != x + 1)
print("range:", lo, hi, "width", hi - lo + 1)
print("images satisfying the constraint:", len(sat))
print("  of those inside the returned range:", sum(lo <= a <= hi for a in sat))
print("images inside the range:", len(inr), "| of those satisfying:", sum(inr))
print("the satisfying set is", runs, "disjoint runs of consecutive addresses")
assert sum(lo <= a <= hi for a in sat) < len(sat)

range: 47802 48314 width 513
images satisfying the constraint: 4096
  of those inside the returned range: 129
images inside the range: 513 | of those satisfying: 129
the satisfying set is 512 disjoint runs of consecutive addresses


**Why:** pixel values are laid out pixel-major, band-minor, and the address is lexicographic in that digit order. Fixing band 0 at every pixel fixes digits scattered through the whole string, including the leading ones, so the satisfying set is a product set spread over many intervals. A correct search returns a union of intervals (or reorders the digits so the constrained band leads).

## 2. `index_image` caches by the first 16 values

In [3]:
s = hg.ImageSpec(8, 8, hg.PixelMode.GRAY8); G = hg.HyperGallery(s)
a = [0]*64; b = [0]*64; b[63] = 200                       # differ only in the last pixel
ra = G.index_image(a, "a"); rb = G.index_image(b, "b")
print("labels equal:", ra.label == rb.label)
print("regenerate(b) == b:", G.regenerate_image(rb) == b, "| regenerate(b) == a:", G.regenerate_image(rb) == a)
assert ra.label == rb.label

labels equal: True
regenerate(b) == b: False | regenerate(b) == a: True
